In [1]:
import pandas as pd

#setando DFs
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

ano_analise = 2025

caminho_entrada = f'input/{ano_analise}_Viagem.csv'
caminho_saida = f'output/tabela_final_{ano_analise}.csv'

# lendo arquivos
df_viagens = pd.read_csv(caminho_entrada, encoding='latin1', sep=';')

# função para mudar formato de colunas (str para float ou datetime)
def transformando_colunas(colunas, raiz_dataframe, to_date=False):
    if to_date:
        for coluna in colunas:
            raiz_dataframe[f'{coluna}'] = pd.to_datetime(raiz_dataframe[f'{coluna}'], format='%d/%m/%Y')
            raiz_dataframe[f'{coluna}'].dt.date
    else:
        for coluna in colunas:
            raiz_dataframe[f'{coluna}'] = raiz_dataframe[f'{coluna}'].apply(lambda x: float(x.replace(',', '.')))

# transformando colunas de valores monetários em float
transformando_colunas(colunas=['Valor diárias','Valor passagens', 'Valor devolução', 'Valor outros gastos'], raiz_dataframe=df_viagens)

# criando nova coluna de despesas totais
df_viagens['Valor total'] = df_viagens['Valor diárias'] + df_viagens['Valor passagens'] + df_viagens['Valor outros gastos']

# ajustando valores NaN
df_viagens['Cargo'] = df_viagens['Cargo'].fillna('NAO IDENTIFICADO')

# transformando colunas em datetime
transformando_colunas(colunas=['Período - Data de início', 'Período - Data de fim'], raiz_dataframe=df_viagens, to_date=True)

# definindo mes da viagem
df_viagens['Mes da viagem'] = df_viagens['Período - Data de início'].dt.month_name()

# definindo duracao em dias da viagem
df_viagens['Dias da viagem'] = (df_viagens['Período - Data de fim'] - df_viagens['Período - Data de início']).dt.days

# criando tabelas consolidadas
df_tabela_consolidada = (
    df_viagens
    .groupby('Cargo')
    .agg(
        despesa_media=('Valor total', 'mean'), 
        duracao_media=('Dias da viagem', 'mean'),
        despesas_totais=('Valor total', 'sum'),
        destinos_mais_frequentes=('Destinos', lambda x: x.value_counts().idxmax() if not x.empty else None),
        n_viagens=('Nome', 'count')
        )
    .reset_index()
    .sort_values(by='despesas_totais', ascending=False)
)

# filtrando tabelas consolidadas por cargos relevantes (>1% do total de viagens)
df_viagem_cargo = df_viagens['Cargo'].value_counts(normalize=True).reset_index()
df_cargos_relevantes = df_viagem_cargo.loc[df_viagem_cargo['proportion'] > 0.01, 'Cargo']

# filtro para cargos relevantes na tabela consolidada
filtro = df_tabela_consolidada['Cargo'].isin(df_cargos_relevantes)

# tabela final consolidada e filtrada
df_final=df_tabela_consolidada[filtro].sort_values(by='n_viagens', ascending=False)

df_final.to_csv(caminho_saida, index=False) # exportando tabela final para csv